 1. Audit et nettoyage


In [39]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [23]:
!pip install keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [keras-tuner]


In [35]:
!pip install optuna


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [optuna]


In [26]:
import pandas as pd

# 1. Charger le fichier
df = pd.read_csv("textes_films_horreur_comedie-checkpoint.csv")

# 2. Nettoyer les espaces superflus sur la colonne 'resume'
df['resume'] = df['resume'].str.strip()

# 3. Compter et supprimer les doublons
doublons_avant = df.duplicated().sum()
df = df.drop_duplicates()
print(f"Nombre de doublons supprimés : {doublons_avant}")

print("\nRépartition par catégorie :")
print(df['cible'].value_counts())

# 5. Analyser la longueur des résumés
df['longueur'] = df['resume'].str.len()
print("\nStatistiques sur la longueur des résumés :")
print(df['longueur'].describe())

Nombre de doublons supprimés : 54

Répartition par catégorie :
cible
0    1195
1    1151
Name: count, dtype: int64

Statistiques sur la longueur des résumés :
count    2346.000000
mean      558.701194
std       349.608596
min       201.000000
25%       337.000000
50%       476.500000
75%       678.750000
max      7104.000000
Name: longueur, dtype: float64


 2 — Découpage du jeu de données

In [27]:

from sklearn.model_selection import train_test_split

# 1. Isoler d'abord un jeu de production (par exemple 20% des données mis de côté)
df_reste, df_production = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df['cible']
)

# 2. Séparer le reste en jeu d'apprentissage (train) et jeu de test
df_train, df_test = train_test_split(
    df_reste,
    test_size=0.25,  # 25% de 80% = 20% du total
    random_state=42,
    stratify=df_reste['cible']
)

# 3. Afficher les tailles de chaque jeu pour vérification
print(f"Taille du jeu d'apprentissage (Train) : {len(df_train)} lignes")
print(f"Taille du jeu de test (Test) : {len(df_test)} lignes")
print(f"Taille du jeu de production (Production) : {len(df_production)} lignes")

Taille du jeu d'apprentissage (Train) : 1407 lignes
Taille du jeu de test (Test) : 469 lignes
Taille du jeu de production (Production) : 470 lignes


 3.Représentation du texte (TF-IDF)

In [28]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Initialiser le vectoriseur TF-IDF
vectorizer = TfidfVectorizer(max_features=5000)

# 2. Ajuster et transformer le jeu d'entraînement (Train)
X_train_tfidf = vectorizer.fit_transform(df_train['resume'])

# 3. Transformer uniquement le jeu de test et de production
X_test_tfidf = vectorizer.transform(df_test['resume'])
X_production_tfidf = vectorizer.transform(df_production['resume'])

# 4. Extraire également les étiquettes (cibles) correspondantes
y_train = df_train['cible']
y_test = df_test['cible']
y_production = df_production['cible']

# Vérification des dimensions
print(f"Dimensions de la matrice TF-IDF d'entraînement : {X_train_tfidf.shape}")
print(f"Dimensions de la matrice TF-IDF de test : {X_test_tfidf.shape}")

Dimensions de la matrice TF-IDF d'entraînement : (1407, 5000)
Dimensions de la matrice TF-IDF de test : (469, 5000)


4- Les points de comparaison

In [29]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# Définition des 5 graines différentes pour les tirages au hasard
seeds = [42, 123, 456, 789, 2023]

f1_lr = []
f1_nb = []
f1_nn = []

for seed in seeds:
    # Utiliser une partie de l'apprentissage pour valider (le test reste intact)
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_tfidf, y_train, test_size=0.2, random_state=seed, stratify=y_train
    )

    # 1. Modèle : Régression Logistique
    lr = LogisticRegression(random_state=seed)
    lr.fit(X_tr, y_tr)
    pred_lr = lr.predict(X_val)
    f1_lr.append(f1_score(y_val, pred_lr, average='weighted'))

    # 2. Modèle : Bayes Naïf (MultinomialNB)
    nb = MultinomialNB()
    nb.fit(X_tr, y_tr)
    pred_nb = nb.predict(X_val)
    f1_nb.append(f1_score(y_val, pred_nb, average='weighted'))

    # 3. Modèle : Réseau de neurones de départ (Keras)
    X_tr_dense = X_tr.toarray()
    X_val_dense = X_val.toarray()

    tf.keras.utils.set_random_seed(seed)
    model_nn = Sequential([
        Dense(64, activation='relu', input_shape=(X_tr_dense.shape[1],)),
        Dropout(0.3),
        Dense(1, activation='sigmoid')
    ])
    model_nn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    model_nn.fit(X_tr_dense, y_tr, epochs=5, batch_size=32, verbose=0)

    pred_nn_prob = model_nn.predict(X_val_dense, verbose=0)
    pred_nn = (pred_nn_prob > 0.5).astype(int).flatten()
    f1_nn.append(f1_score(y_val, pred_nn, average='weighted'))

# Calcul des moyennes et de la marge de bruit (écart-type)
print("--- Résultats F1-score sur 5 graines ---")
print(f"Régression Logistique : Moyenne = {np.mean(f1_lr):.4f} | Marge de bruit (std) = {np.std(f1_lr):.4f}")
print(f"Naive Bayes           : Moyenne = {np.mean(f1_nb):.4f} | Marge de bruit (std) = {np.std(f1_nb):.4f}")
print(f"Réseau de neurones    : Moyenne = {np.mean(f1_nn):.4f} | Marge de bruit (std) = {np.std(f1_nn):.4f}")

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr

--- Résultats F1-score sur 5 graines ---
Régression Logistique : Moyenne = 0.8371 | Marge de bruit (std) = 0.0204
Naive Bayes           : Moyenne = 0.8680 | Marge de bruit (std) = 0.0209
Réseau de neurones    : Moyenne = 0.8693 | Marge de bruit (std) = 0.0218


5.Recherche des réglages avec KerasTuner

In [34]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.regularizers import l2
import keras_tuner as kt

def build_model(hp):
    model = Sequential()

    # 1. Réglage : Nombre de couches cachées
    n_layers = hp.Int('n_layers', min_value=1, max_value=3, step=1)

    # 2. Réglage : Nombre de neurones
    units = hp.Int('units', min_value=32, max_value=128, step=32)

    # 3. Réglage : Régularisation (L2) et taux de Dropout
    l2_reg = hp.Choice('l2_reg', [1e-4, 1e-3, 1e-2])
    dropout_rate = hp.Float('dropout_rate', min_value=0.1, max_value=0.5, step=0.1)

    # Première couche avec la forme d'entrée (input_shape) de vos données TF-IDF
    model.add(Dense(
        units=units,
        activation='relu',
        kernel_regularizer=l2(l2_reg),
        input_shape=(X_train_tfidf.shape[1],)
    ))
    model.add(Dropout(dropout_rate))

    # Ajout des couches supplémentaires si n_layers > 1
    for _ in range(n_layers - 1):
        model.add(Dense(units=units, activation='relu', kernel_regularizer=l2(l2_reg)))
        model.add(Dropout(dropout_rate))

    # Couche de sortie pour la classification binaire (0 ou 1)
    model.add(Dense(1, activation='sigmoid'))

    # 5. Réglage : Taux d'apprentissage (Learning rate) de l'optimiseur[cite: 3]
    learning_rate = hp.Choice('learning_rate', [1e-2, 1e-3, 1e-4])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [33]:
import keras_tuner as kt

# Configuration de la recherche aléatoire (RandomSearch) sur 20 essais
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=20,
    executions_per_trial=1,
    directory='keras_tuner_dir',
    project_name='films_classification'
)

# Préparation des données d'entraînement et de validation
X_train_dense = X_train_tfidf.toarray()
X_test_dense = X_test_tfidf.toarray()

# Lancement de la recherche d'hyperparamètres
tuner.search(
    X_train_dense, y_train,
    epochs=5,
    validation_split=0.2,
    verbose=1
)

# Récupération des meilleurs hyperparamètres
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print("\n--- Meilleurs hyperparamètres trouvés ---")
print(f"Nombre de couches cachées : {best_hps.get('n_layers')}")
print(f"Nombre de neurones : {best_hps.get('units')}")
print(f"Taux de Dropout : {best_hps.get('dropout_rate')}")
print(f"Régularisation L2 : {best_hps.get('l2_reg')}")
print(f"Taux d'apprentissage : {best_hps.get('learning_rate')}")

Trial 20 Complete [00h 00m 04s]
val_accuracy: 0.7695035338401794

Best val_accuracy So Far: 0.8865247964859009
Total elapsed time: 00h 01m 35s

--- Meilleurs hyperparamètres trouvés ---
Nombre de couches cachées : 2
Nombre de neurones : 32
Taux de Dropout : 0.5
Régularisation L2 : 0.0001
Taux d'apprentissage : 0.001


6.Recherche des réglages avec Optuna

In [36]:
import optuna
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.regularizers import l2

def objective(trial):
    # Suggestion des 5 hyperparamètres
    n_layers = trial.suggest_int('n_layers', 1, 3)
    units = trial.suggest_int('units', 32, 128, step=32)
    l2_reg = trial.suggest_categorical('l2_reg', [1e-4, 1e-3, 1e-2])
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5, step=0.1)
    learning_rate = trial.suggest_categorical('learning_rate', [1e-2, 1e-3, 1e-4])

    # Construction du modèle
    model = Sequential()
    model.add(Dense(
        units=units,
        activation='relu',
        kernel_regularizer=l2(l2_reg),
        input_shape=(X_train_dense.shape[1],)
    ))
    model.add(Dropout(dropout_rate))

    for _ in range(n_layers - 1):
        model.add(Dense(units=units, activation='relu', kernel_regularizer=l2(l2_reg)))
        model.add(Dropout(dropout_rate))

    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    # Entraînement avec validation split pour évaluer l'essai
    history = model.fit(
        X_train_dense, y_train,
        epochs=5,
        batch_size=32,
        validation_split=0.2,
        verbose=0
    )

    # On retourne la meilleure précision de validation de cet essai
    val_acc = max(history.history['val_accuracy'])
    return val_acc

# Lancement de l'étude Optuna (exactement 20 essais)
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

print("\n--- Meilleurs hyperparamètres trouvés par Optuna ---")
print(study.best_params)
print(f"Meilleur score (val_accuracy) : {study.best_value:.4f}")

[I 2026-10-26 11:00:29,100] A new study created in memory with name: no-name-4844384e-77a4-48c5-8745-10d5ee5903ea
/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
[I 2026-10-26 11:00:32,714] Trial 0 finished with value: 0.8829787373542786 and parameters: {'n_layers': 2, 'units': 64, 'l2_reg': 0.0001, 'dropout_rate': 0.2, 'learning_rate': 0.001}. Best is trial 0 with value: 0.8829787373542786.
[I 2026-10-26 11:00:35,120] Trial 1 finished with value: 0.8297872543334961 and parameters: {'n_layers': 1, 'units': 128, 'l2_reg': 0.01, 'dropout_rate': 0.5, 'learning_rate': 0.01}. Best is trial 0 with value: 0.8829787373542786.
[I 2026-10-26 11:00:38,467] Trial 2 finished with value: 0.8794326186180115 and parameters


--- Meilleurs hyperparamètres trouvés par Optuna ---
{'n_layers': 2, 'units': 128, 'l2_reg': 0.001, 'dropout_rate': 0.4, 'learning_rate': 0.001}
Meilleur score (val_accuracy) : 0.8901


7.Verdict sur le test

In [37]:

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.regularizers import l2
from sklearn.metrics import f1_score, classification_report, accuracy_score

# Meilleurs hyperparamètres (issus d'Optuna)
best_params = {
    'n_layers': 2,
    'units': 128,
    'l2_reg': 0.001,
    'dropout_rate': 0.4,
    'learning_rate': 0.001
}

# 1. Construction du modèle final
model_final = Sequential()
model_final.add(Dense(
    units=best_params['units'],
    activation='relu',
    kernel_regularizer=l2(best_params['l2_reg']),
    input_shape=(X_train_dense.shape[1],)
))
model_final.add(Dropout(best_params['dropout_rate']))

for _ in range(best_params['n_layers'] - 1):
    model_final.add(Dense(units=best_params['units'], activation='relu', kernel_regularizer=l2(best_params['l2_reg'])))
    model_final.add(Dropout(best_params['dropout_rate']))

model_final.add(Dense(1, activation='sigmoid'))

# Compilation
model_final.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=best_params['learning_rate']),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# 2. Entraînement sur l'intégralité du jeu d'entraînement
print("Entraînement du modèle final sur tout le train...")
model_final.fit(
    X_train_dense, y_train,
    epochs=5,
    batch_size=32,
    verbose=1
)

# 3. Évaluation finale sur le jeu de test (totalement intact)
print("\n--- Résultats finaux sur le jeu de test ---")
y_pred_prob = model_final.predict(X_test_dense)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

test_accuracy = accuracy_score(y_test, y_pred)
test_f1 = f1_score(y_test, y_pred, average='weighted')

print(f"Exactitude (Accuracy) sur le test : {test_accuracy:.4f}")
print(f"Score F1 pondéré sur le test : {test_f1:.4f}\n")

print("Rapport de classification détaillé :")
print(classification_report(y_test, y_pred))

Entraînement du modèle final sur tout le train...
Epoch 1/5
44/44 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.5999 - loss: 0.8600
Epoch 2/5
44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.8621 - loss: 0.5954
Epoch 3/5
44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.9566 - loss: 0.3099
Epoch 4/5
44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.9922 - loss: 0.2108
Epoch 5/5
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9993 - loss: 0.1723

--- Résultats finaux sur le jeu de test ---
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
Exactitude (Accuracy) sur le test : 0.8465
Score F1 pondéré sur le test : 0.8456

Rapport de classification détaillé :
              precision    recall  f1-score   support

           0       0.81      0.92      0.86       239
           1       0.90      0.77      0.83       230

    accuracy                           0.85       469
   macro avg       0.85      0.85      0.85       469
weighted avg       0.85      0.85      0.85       469



8. Sauvegarde

In [41]:
import os
import json
import joblib
import tensorflow as tf
import sklearn
import numpy as np

# 1. Créer le dossier 'model' s'il n'existe pas
os.makedirs('model', exist_ok=True)

# 2. Sauvegarder le réseau de neurones au format .keras
model_final.save('model/model.keras')

# 3. Sauvegarder le vectoriseur TF-IDF avec joblib
joblib.dump(vectorizer, 'model/vectoriseur.joblib')

# 4. Créer et écrire le fichier metrics.json (en encodage UTF-8)
metrics = {
    "model_version": "1.0",
    "test_f1_score": float(test_f1),
    "test_size": int(len(y_test)),
    "library_versions": {
        "tensorflow": tf.__version__,
        "scikit-learn": sklearn.__version__,
        "numpy": np.version.version
    }
}

with open('model/metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, ensure_ascii=False, indent=4)

# 5. Sauvegarder un petit échantillon de données de référence
reference_sample = df.head(10)
reference_sample.to_csv('model/reference.csv', index=False)

print(" Tous les artefacts ont été sauvegardés avec succès dans le dossier 'model/' !")

 Tous les artefacts ont été sauvegardés avec succès dans le dossier 'model/' !
